# CRSP Data Pull & Sanity Checks

Pull CRSP monthly stock file and run basic sanity checks before using in analysis.

**Run 00_setup_check.ipynb first.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from fnce_research.data.crsp import load_msf, load_msenames, load_msedelist

START_YR, END_YR = 1963, 2023

In [ ]:
# Pull MSF (checks cache first)
msf = load_msf(START_YR, END_YR)
print(f'MSF: {msf.shape[0]:,} rows, {msf.shape[1]} cols')
print(f'Date range: {msf.date.min()} — {msf.date.max()}')
msf.head()

In [ ]:
# Sanity: return distribution
print(msf['ret'].describe())
print(f'Missing ret: {msf["ret"].isna().mean():.1%}')

In [ ]:
# Stock universe size over time
n_stocks = msf.groupby('date')['permno'].nunique()
n_stocks.plot(title='Number of stocks in CRSP MSF by month', figsize=(12, 4))
plt.ylabel('N stocks')
plt.tight_layout()
plt.show()

In [ ]:
# Load names for exchange/share-type filters
names = load_msenames()
print(f'msenames: {names.shape}')

# Common stock (shrcd 10/11) on major exchanges (exchcd 1/2/3)
common = names[names['shrcd'].isin([10, 11]) & names['exchcd'].isin([1, 2, 3])]
print(f'Common stock observations: {len(common):,}')

In [ ]:
# Delisting returns (critical for avoiding survivorship bias)
delist = load_msedelist()
print(f'Delisting observations: {len(delist):,}')
print(delist['dlstcd'].value_counts().head(10))